# 21 — Conference extension of the evaluation-protocol study

Rebuilds every number for the conference version of the June 2026 preprint (Preprints.org 202606.0903)
from the preprint's data only (CSE-CIC-IDS2018 and CIC-DDoS2019, CICFlowMeter features); the NetFlow-v2
corpora and the four-corpus study are not used.

Design log
* Three models (Random Forest, LightGBM, MLP) and three seeds in every experiment.
* Within-dataset: random split, time split with every class seen, global time split with later attacks
  unseen; class-by-class scores plus attack-or-benign alarm rates, per class.
* Transfer: zero-shot and Platt or isotonic recalibration on 1, 5 and 10% time-ordered labelled buffers per
  capture day; AUROC/AUPRC (ranking) beside MCC (decisions); zero-shot also scored on the rows the
  recalibrated model is scored on.
* Per-attack-type ranking per capture day; drift per capture day for all, benign and attack traffic;
  label audit of CSE-CIC-IDS2018; reproduction check against the preprint's committed tables.
* CSE-CIC-IDS2018 dates are day first and are parsed with an explicit format.
* The processed CIC-DDoS2019 file uses the CSE-CIC-IDS2018 short feature names; the mapping to the
  cleaned CIC-DDoS2019 names is recovered from regime 1 by exact value matching and applied to regime 2.
* Compute: CPU. No step uses a GPU (scikit-learn and LightGBM CPU builds). Model fits: 27 within-dataset,
  6 label-merge sensitivity, 9 transfer, 1 reproduction. No model API calls.
* Stages 3 and 4 append to their CSVs and skip finished (model, split or day, seed) cells.

In [ ]:
# ---- Setup -------------------------------------------------------------------
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

BASE = os.environ.get('DRIFT_BASE', '/content/drive/MyDrive/drift-conference')
DATA = f'{BASE}/data'
OUT  = f'{BASE}/results/conference'
FIG  = f'{OUT}/figures'
os.makedirs(FIG, exist_ok=True)
os.environ['DRIFT_DATA_ROOT'] = DATA
sys.path.insert(0, f'{BASE}/src')

import time, gc, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score, average_precision_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import BaseEstimator, ClassifierMixin
from lightgbm import LGBMClassifier
import sklearn, lightgbm

from driftbench.splits import random_split
from driftbench.harmonise import harmonise_labels, shared_families
from driftbench.metrics import expected_calibration_error
from driftbench.drift import psi, ks_stat
from driftbench.models import time_ordered_buffer

CFG = dict(seeds=[0, 1, 2], models=['rf', 'lgbm', 'mlp'],
           splits=['random', 'time_all_seen', 'time_global'],
           variant_models=['rf', 'lgbm'],
           fracs=[0.01, 0.05, 0.10], methods=['sigmoid', 'isotonic'],
           source_train_frac=0.8, drift_sample=300_000, native_min_rows=50,
           ece_bins=15, dpi=600)
MODEL_NAME = {'rf': 'Random Forest', 'lgbm': 'LightGBM', 'mlp': 'MLP'}
SPLIT_NAME = {'random': 'Random split',
              'time_all_seen': 'Time split, all classes seen',
              'time_global': 'Time split, later attacks unseen'}
METHOD_NAME = {'none': 'No recalibration', 'sigmoid': 'Platt', 'isotonic': 'Isotonic'}
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| pandas', pd.__version__)
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=1))

## Stage 1 — Load the data and check it

Missing data files raise an error. CSE-CIC-IDS2018 dates are day first (14/02/2018) and are parsed
with that explicit format; more than 0.5% unparsed dates raise an error. The capture-day schedule is
printed. CIC-DDoS2019 has two capture days in two folders: `CIC-DDoS2019` (1 December 2018,
regime 1) and `CIC-DDoS2019-test` (3 November 2018, regime 2, the day with Portmap). Both are reduced
to the same common-core features used for CSE-CIC-IDS2018.

In [ ]:
P_IDS = f'{DATA}/interim/CSE-CIC-IDS2018'
P_R1, P_R2 = f'{DATA}/interim/CIC-DDoS2019', f'{DATA}/interim/CIC-DDoS2019-test'
need = {
    'IDS2018 features': f'{P_IDS}/features.parquet', 'IDS2018 labels': f'{P_IDS}/labels.parquet',
    'IDS2018 metadata': f'{P_IDS}/metadata.parquet',
    'IDS2018 core features': f'{DATA}/processed/CSE-CIC-IDS2018/features_core.parquet',
    'IDS2018 family labels': f'{DATA}/processed/CSE-CIC-IDS2018/labels_family.parquet',
    'DDoS2019 day 1 core features (processed)': f'{DATA}/processed/CIC-DDoS2019/features_core.parquet',
    'DDoS2019 day 1 features': f'{P_R1}/features.parquet', 'DDoS2019 day 1 labels': f'{P_R1}/labels.parquet',
    'DDoS2019 day 1 metadata': f'{P_R1}/metadata.parquet',
    'DDoS2019 day 2 features': f'{P_R2}/features.parquet', 'DDoS2019 day 2 labels': f'{P_R2}/labels.parquet',
    'DDoS2019 day 2 metadata': f'{P_R2}/metadata.parquet',
}
missing = [f'{k}: {p}' for k, p in need.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing data files:\n' + '\n'.join(missing))

def time_col(meta):
    for c in meta.columns:
        if 'timestamp' in c or 'date' in c:
            return c
    raise KeyError(f'no time column in {list(meta.columns)}')

def parse_time(series, fmt, dayfirst):
    s = series.astype(str).str.strip()
    t = pd.to_datetime(s, format=fmt, errors='coerce')
    bad = t.isna()
    if bad.any():   # second pass for rows written in another layout (e.g. without seconds)
        t[bad] = pd.to_datetime(s[bad], format='mixed', dayfirst=dayfirst, errors='coerce')
    return t

def is_benign(arr):
    return pd.Series(arr).astype(str).str.lower().str.contains('benign|normal').to_numpy()

# --- CSE-CIC-IDS2018, native features and labels (within-dataset study) ---
XA_df = pd.read_parquet(need['IDS2018 features'])
yA = pd.read_parquet(need['IDS2018 labels'])['label'].astype(str).str.strip().to_numpy()
mA = pd.read_parquet(need['IDS2018 metadata'])
assert len(XA_df) == len(yA) == len(mA), 'IDS2018 files are not row-aligned'
XA = XA_df.to_numpy(np.float32)
FEAT_A = list(XA_df.columns)
tA = parse_time(mA[time_col(mA)], '%d/%m/%Y %H:%M:%S', dayfirst=True)
n_bad = int(tA.isna().sum())
print(f'IDS2018: {len(yA):,} flows, {len(FEAT_A)} features, {len(np.unique(yA))} classes, '
      f'unparsed timestamps {n_bad:,} ({n_bad / len(yA):.4%}); first raw value {mA[time_col(mA)].iloc[0]!r}')
assert n_bad / len(yA) < 0.005, 'too many unparsed IDS2018 timestamps: the time splits would be wrong'
ORDER_A = np.argsort(tA.to_numpy(), kind='stable')   # unparsed rows (if any) sort last
sched = (pd.DataFrame({'day': tA.dt.date.astype(str), 'label': yA})
         .groupby('day')['label'].agg(lambda s: ', '.join(f'{k} ({v:,})' for k, v in s.value_counts().items())))
print('\nCapture-day schedule as read (CIC: 14-16 Feb brute force and DoS, 20-21 Feb DDoS, 22-23 Feb web, '
      '28 Feb-1 Mar infiltration, 2 Mar bot):')
for d, s_ in sched.items():
    print(f'  {d}: {s_}')

# --- Cross-dataset views (common-core features, family labels) ---
XS_df = pd.read_parquet(need['IDS2018 core features'])
yS = pd.read_parquet(need['IDS2018 family labels'])['family'].astype(str).to_numpy()
CORE = list(XS_df.columns)
P1 = pd.read_parquet(need['DDoS2019 day 1 core features (processed)'])
assert list(P1.columns) == CORE, 'processed DDoS2019 core columns differ from IDS2018 core columns'
# The processed files carry the 2018 short names (e.g. fwd_pkt_len_max). The cleaned CIC-DDoS2019 files
# use the long names (e.g. fwd_packet_length_max). Recover the exact column mapping from regime 1, where
# the processed file and the cleaned file hold the same rows: each processed column is matched to the
# cleaned column with identical values. The mapping is then checked on every row and applied to regime 2.
import difflib
F1 = pd.read_parquet(need['DDoS2019 day 1 features'])
assert len(F1) == len(P1), 'regime 1 processed and cleaned files differ in length'
n_chk = min(200_000, len(P1))
Pv = P1.iloc[:n_chk].to_numpy(np.float64); Fv = F1.iloc[:n_chk].to_numpy(np.float64); fcols = list(F1.columns)
COLMAP, ambiguous = {}, {}
for j, c in enumerate(CORE):
    eq = [fcols[k] for k in range(Fv.shape[1])
          if np.allclose(Fv[:, k], Pv[:, j], rtol=1e-6, atol=0, equal_nan=True)]
    if not eq:
        raise ValueError(f'no cleaned CIC-DDoS2019 column matches processed column {c}')
    COLMAP[c] = c if c in eq else max(eq, key=lambda x: difflib.SequenceMatcher(None, x, c).ratio())
    if len(eq) > 1:
        ambiguous[c] = eq
full_ok = np.allclose(F1[[COLMAP[c] for c in CORE]].to_numpy(np.float32), P1.to_numpy(np.float32), equal_nan=True)
print(f'column mapping recovered for {len(COLMAP)} core features; holds on every regime 1 row: {full_ok}')
assert full_ok
renamed = {c: m for c, m in COLMAP.items() if c != m}
print('renamed:', ', '.join(f'{c} <- {m}' for c, m in renamed.items()))
del F1, Pv, Fv; _ = gc.collect()

day = {}
for r, p in [('regime_1', P_R1), ('regime_2', P_R2)]:
    f = pd.read_parquet(f'{p}/features.parquet')
    miss = [COLMAP[c] for c in CORE if COLMAP[c] not in f.columns]
    assert not miss, f'{r}: cleaned features missing {miss}'
    for c, cands in ambiguous.items():   # columns that were identical in regime 1: are they still identical here?
        present = [x for x in cands if x in f.columns]
        same = all(np.allclose(f[present[0]].to_numpy(np.float64), f[x].to_numpy(np.float64), equal_nan=True) for x in present[1:])
        if r == 'regime_2' or not same:
            print(f'{r}: {c} matched identical columns {present}; still identical here: {same}; using {COLMAP[c]}')
    f = f[[COLMAP[c] for c in CORE]].set_axis(CORE, axis=1)
    lab = pd.read_parquet(f'{p}/labels.parquet')['label'].astype(str).str.strip().to_numpy()
    met = pd.read_parquet(f'{p}/metadata.parquet')
    assert len(f) == len(lab) == len(met), f'{r} files are not row-aligned'
    t = parse_time(met[time_col(met)], '%Y-%m-%d %H:%M:%S.%f', dayfirst=False)
    day[r] = dict(X=f[CORE].to_numpy(np.float32), native=lab, t=t.to_numpy(),
                  days=sorted(t.dt.date.astype(str).unique()), unparsed=int(t.isna().sum()))
    del f
del P1
FAMS = shared_families(pd.Series(yS), pd.Series(np.concatenate([day[r]['native'] for r in day])))
print('common-core features:', len(CORE), '| shared families:', FAMS)
assert any(pd.Series(day['regime_2']['native']).str.lower().str.contains('portmap')), 'regime_2 has no Portmap'

XT_parts, yT_parts, nT_parts, tT_parts, REG_ROWS, off = [], [], [], [], {}, 0
for r in ['regime_1', 'regime_2']:
    fam = harmonise_labels(pd.Series(day[r]['native'])).astype(str).to_numpy()
    keep = np.isin(fam, FAMS)
    XT_parts.append(day[r]['X'][keep]); yT_parts.append(fam[keep]); nT_parts.append(day[r]['native'][keep])
    tT_parts.append(day[r]['t'][keep]); REG_ROWS[r] = np.arange(off, off + keep.sum()); off += keep.sum()
    vc = pd.Series(fam[keep]).value_counts()
    print(f"{r}: capture day {day[r]['days']}, unparsed timestamps {day[r]['unparsed']}, "
          f"{keep.sum():,} of {len(keep):,} rows kept (shared families) | " +
          ', '.join(f'{k} {v:,}' for k, v in vc.items()) +
          f" | dropped types: {sorted(set(day[r]['native'][~keep]))}")
XT, yT, nT, tT = np.vstack(XT_parts), np.concatenate(yT_parts), np.concatenate(nT_parts), np.concatenate(tT_parts)
keepS = np.isin(yS, FAMS)
XS = XS_df.to_numpy(np.float32)[keepS]; yS = yS[keepS]
print(f'source (IDS2018, shared families): {len(yS):,} rows |', pd.Series(yS).value_counts().to_dict())
del XA_df, XS_df, XT_parts, day; _ = gc.collect()

## Stage 2 — Label audit of CSE-CIC-IDS2018

Finds feature vectors that appear under more than one label. Such rows cannot be told apart by
any model, so they set a ceiling on per-class scores. Writes `conf_label_audit.csv` (per label)
and `conf_label_pairs.csv` (which labels share vectors).

In [ ]:
h = pd.util.hash_pandas_object(pd.DataFrame(XA), index=False).to_numpy()
d = pd.DataFrame({'h': h, 'y': yA})
nlab = d.groupby('h')['y'].nunique()
multi = set(nlab[nlab > 1].index)
d['shared'] = d['h'].isin(multi)
aud = d.groupby('y').agg(n=('h', 'size'), n_shared=('shared', 'sum')).reset_index().rename(columns={'y': 'label'})
aud['share'] = aud['n_shared'] / aud['n']
sets = d[d['shared']].groupby('h')['y'].agg(lambda s: tuple(sorted(set(s))))
pairs = {}
cnt = d[d['shared']].groupby(['h', 'y']).size()
for hh, labs in sets.items():
    for i in range(len(labs)):
        for j in range(i + 1, len(labs)):
            k = (labs[i], labs[j])
            pairs[k] = pairs.get(k, 0) + int(cnt[(hh, labs[i])] + cnt[(hh, labs[j])])
pairs_df = pd.DataFrame([dict(label_a=a, label_b=b, rows=v) for (a, b), v in pairs.items()]).sort_values('rows', ascending=False) \
    if pairs else pd.DataFrame(columns=['label_a', 'label_b', 'rows'])
aud.to_csv(f'{OUT}/conf_label_audit.csv', index=False)
pairs_df.to_csv(f'{OUT}/conf_label_pairs.csv', index=False)
print(aud.sort_values('share', ascending=False).to_string(index=False))
print('\nlabel pairs sharing identical feature vectors:'); print(pairs_df.head(10).to_string(index=False))
del d, h; _ = gc.collect()

## Helpers: models, metrics, resumable CSVs

`panel` reproduces the package's `compute_metrics` exactly but runs vectorised on millions of rows
(checked against it in Stage 3). Recalibration is fitted on the frozen model's stored probabilities,
which is mathematically the same as `driftbench.models.recalibrate` and avoids re-scoring the target.

In [ ]:
def make_model(name, seed):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed,
                                      class_weight='balanced_subsample')
    if name == 'lgbm':
        return LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                              class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    if name == 'mlp':
        return EncodedLabels(Pipeline([('scaler', StandardScaler()),
                                       ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200,
                                                             early_stopping=True, validation_fraction=0.1,
                                                             n_iter_no_change=10, random_state=seed))]))
    raise ValueError(name)

class EncodedLabels(ClassifierMixin, BaseEstimator):
    # Fits the wrapped model on integer class codes and reports the original class names:
    # in scikit-learn 1.6 and 1.7, MLP early stopping fails on string labels.
    def __init__(self, model=None):
        self.model = model
    def fit(self, X, y):
        self.classes_, codes = np.unique(np.asarray(y).astype(str), return_inverse=True)
        self.model.fit(X, codes)
        return self
    def predict_proba(self, X):
        return self.model.predict_proba(X)
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

class StoredProba(ClassifierMixin, BaseEstimator):
    # Passes stored probabilities through, so CalibratedClassifierCV can calibrate them.
    def __init__(self, classes=None):
        self.classes = classes
    def fit(self, X=None, y=None):
        self.classes_ = np.asarray(self.classes); return self
    def predict_proba(self, X):
        return np.asarray(X, float)
    def predict(self, X):
        return self.classes_[np.asarray(X).argmax(1)]

def calibrate(P_buf, y_buf, P_eval, classes, method):
    try:
        from sklearn.frozen import FrozenEstimator
        cal = CalibratedClassifierCV(FrozenEstimator(StoredProba(classes).fit()), method=method)
    except ImportError:
        cal = CalibratedClassifierCV(StoredProba(classes).fit(), method=method, cv='prefit')
    cal.fit(P_buf, y_buf)
    assert list(cal.classes_) == list(classes)
    return cal.predict_proba(P_eval)

def panel(y_true, P, classes):
    # Same definitions as driftbench.metrics.compute_metrics, plus attack-vs-benign scores.
    y_true = np.asarray(y_true).astype(str); classes = np.asarray(classes).astype(str)
    pred = classes[P.argmax(1)]
    m = dict(macro_f1=f1_score(y_true, pred, average='macro', zero_division=0),
             weighted_f1=f1_score(y_true, pred, average='weighted', zero_division=0),
             mcc=matthews_corrcoef(y_true, pred))
    Y = label_binarize(y_true, classes=list(classes))
    if Y.shape[1] == 1:
        Y = np.hstack([1 - Y, Y])
    try:
        m['auroc'] = roc_auc_score(Y, P, average='macro', multi_class='ovr')
    except ValueError:
        m['auroc'] = np.nan
    try:
        m['auprc'] = average_precision_score(Y, P, average='macro')
    except ValueError:
        m['auprc'] = np.nan
    m['brier'] = float(np.mean(np.sum((P - Y) ** 2, axis=1)))
    m['ece'] = expected_calibration_error(y_true, pred, P.max(axis=1), n_bins=CFG['ece_bins'])
    ben_col = np.where(is_benign(classes))[0]
    assert len(ben_col) == 1, f'benign column not unique in {classes}'
    t_att = ~is_benign(y_true); p_att = P.argmax(1) != ben_col[0]; s_att = 1.0 - P[:, ben_col[0]]
    m['n_benign'] = int((~t_att).sum()); m['n_attack'] = int(t_att.sum())
    m['fp_rate'] = float(p_att[~t_att].mean()) if (~t_att).any() else np.nan
    m['attack_recall'] = float(p_att[t_att].mean()) if t_att.any() else np.nan
    if t_att.any() and (~t_att).any():
        m['bin_auroc'] = roc_auc_score(t_att, s_att); m['bin_auprc'] = average_precision_score(t_att, s_att)
    else:
        m['bin_auroc'] = m['bin_auprc'] = np.nan
    return m

METRICS = ['macro_f1', 'weighted_f1', 'mcc', 'auroc', 'auprc', 'brier', 'ece', 'n_benign', 'n_attack',
           'fp_rate', 'attack_recall', 'bin_auroc', 'bin_auprc']

def done_keys(path, keys):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    return set(map(tuple, df[keys].astype(str).to_numpy()))

def append(path, rows, cols):
    df = pd.DataFrame(rows)[cols]
    df.to_csv(path, mode='a', header=not os.path.exists(path), index=False)

## Stage 3 — Within-dataset study (CSE-CIC-IDS2018)

Three splits, three models, three seeds:
* **Random split**: stratified 60/20/20, as most papers report.
* **Time split, all classes seen**: each class is split in time order, earliest 60% to training and
  latest 20% to test, so every class appears on both sides.
* **Time split, later attacks unseen**: the whole capture is split in time order 60/20/20, so attacks
  that only happen late in the capture are absent from training.

Sensitivity variant: FTP-BruteForce and DoS-SlowHTTPTest merged into one label (random split,
Random Forest and LightGBM), because the CIC capture recorded these two incorrectly (Stage 2 shows the
overlap). Writes `conf_demoA.csv` (one row per fit) and `conf_demoA_classes.csv` (one row per class).

In [ ]:
def split_indices(name, seed, y):
    if name == 'random':
        s = random_split(len(y), pd.Series(y), seed=seed)
        return s['train'], s['test']
    if name == 'time_all_seen':
        tr, te = [], []
        yo = y[ORDER_A]
        for c in np.unique(y):
            idx = ORDER_A[yo == c]
            a, b = int(len(idx) * 0.6), int(len(idx) * 0.8)
            tr.append(idx[:a]); te.append(idx[b:])
        return np.concatenate(tr), np.concatenate(te)
    if name == 'time_global':
        a, b = int(len(y) * 0.6), int(len(y) * 0.8)
        return ORDER_A[:a], ORDER_A[b:]
    raise ValueError(name)

FTP, SLOW = 'FTP-BruteForce', 'DoS attacks-SlowHTTPTest'
MERGED = 'FTP-BruteForce + DoS-SlowHTTPTest'
VARIANTS = {'native': yA}
if FTP in set(yA) and SLOW in set(yA):
    VARIANTS['merged'] = np.where(np.isin(yA, [FTP, SLOW]), MERGED, yA)
else:
    print('merged variant skipped: labels not found')

A_CSV, A_CLS = f'{OUT}/conf_demoA.csv', f'{OUT}/conf_demoA_classes.csv'
A_KEYS = ['variant', 'model', 'split', 'seed']
A_COLS = A_KEYS + ['n_train', 'n_test', 'n_test_unseen'] + METRICS + ['fit_s']
C_COLS = A_KEYS + ['label', 'n_test', 'in_train', 'recall', 'flagged_attack', 'top_pred', 'top_pred_share']
jobs = [('native', m, s, sd) for m in CFG['models'] for s in CFG['splits'] for sd in CFG['seeds']]
if 'merged' in VARIANTS:
    jobs += [('merged', m, 'random', sd) for m in CFG['variant_models'] for sd in CFG['seeds']]
done = done_keys(A_CSV, A_KEYS)
checked = False
for (var, mname, sname, seed) in jobs:
    if (var, mname, sname, str(seed)) in done:
        continue
    y = VARIANTS[var]
    tr, te = split_indices(sname, seed, y)
    t0 = time.time()
    mdl = make_model(mname, seed).fit(XA[tr], y[tr])
    fit_s = time.time() - t0
    P = mdl.predict_proba(XA[te]); classes = np.asarray(mdl.classes_).astype(str)
    yt = y[te]; m = panel(yt, P, classes)
    if not checked:
        from driftbench.metrics import compute_metrics
        ref = compute_metrics(yt, classes[P.argmax(1)], P, classes)
        for k in ('macro_f1', 'mcc', 'auroc', 'auprc', 'brier', 'ece', 'fp_rate'):
            a, b = m[k], ref[k]
            assert (np.isnan(a) and np.isnan(b)) or abs(a - b) < 1e-9, f'panel differs on {k}: {a} vs {b}'
        checked = True; print('panel matches driftbench.compute_metrics')
    seen = set(y[tr])
    row = dict(variant=var, model=mname, split=sname, seed=seed, n_train=len(tr), n_test=len(te),
               n_test_unseen=int((~np.isin(yt, list(seen))).sum()), fit_s=round(fit_s, 1), **m)
    pred = classes[P.argmax(1)]; flag = ~is_benign(pred)
    crow = []
    for c in np.unique(yt):
        msk = yt == c; vc = pd.Series(pred[msk]).value_counts()
        crow.append(dict(variant=var, model=mname, split=sname, seed=seed, label=c, n_test=int(msk.sum()),
                         in_train=c in seen, recall=float((pred[msk] == c).mean()),
                         flagged_attack=float(flag[msk].mean()), top_pred=vc.index[0],
                         top_pred_share=float(vc.iloc[0] / msk.sum())))
    append(A_CSV, [row], A_COLS); append(A_CLS, crow, C_COLS)
    print(f"{var:7s} {mname:5s} {sname:14s} seed {seed}: macro-F1 {m['macro_f1']:.3f} | attack recall "
          f"{m['attack_recall']:.3f} | benign FPR {m['fp_rate']:.3f} | fit {fit_s:.0f}s")
    del mdl, P; gc.collect()
print('Stage 3 complete')

## Stage 4 — Cross-dataset transfer (CSE-CIC-IDS2018 to CIC-DDoS2019)

For each model and seed: train on a stratified 80% of the source (shared families only), score the
held-out 20% (in-domain reference), and score every target row once. Then, per capture day, take the
first 1, 5 or 10% of that day's flows in time order as a labelled buffer, recalibrate the frozen model
on it (Platt or isotonic), and score the rest of the day. Zero-shot scores are also computed on the
same remaining rows (method `none`). Writes `conf_demoB.csv` and `conf_demoB_types.csv`
(per CIC-DDoS2019 attack type: alarm rate and AUROC against that day's benign flows).

In [ ]:
B_CSV, B_TYP = f'{OUT}/conf_demoB.csv', f'{OUT}/conf_demoB_types.csv'
B_KEYS = ['model', 'seed', 'regime', 'method', 'frac']
B_COLS = B_KEYS + ['n_eval', 'n_buffer', 'buffer_benign', 'single_class_buffer'] + METRICS + ['fit_s']
T_COLS = B_KEYS + ['attack_type', 'n', 'flagged_attack', 'auroc_vs_benign']
ORD = {r: rows[np.argsort(tT[rows], kind='stable')] for r, rows in REG_ROWS.items()}

def type_rows(base, nat, yv, P, classes):
    ben_col = np.where(is_benign(classes))[0][0]
    s = 1 - P[:, ben_col]; ben = is_benign(yv); out = []
    for t in sorted(set(nat[~ben])):
        msk = (nat == t) & ~ben
        if msk.sum() < CFG['native_min_rows'] or ben.sum() < CFG['native_min_rows']:
            continue
        lab = np.r_[np.ones(msk.sum()), np.zeros(ben.sum())]
        out.append(dict(**base, attack_type=t, n=int(msk.sum()),
                        flagged_attack=float((P[msk].argmax(1) != ben_col).mean()),
                        auroc_vs_benign=roc_auc_score(lab, np.r_[s[msk], s[ben]])))
    return out

done = done_keys(B_CSV, B_KEYS)
for mname in CFG['models']:
    for seed in CFG['seeds']:
        if (mname, str(seed), 'regime_2', 'isotonic', str(CFG['fracs'][-1])) in done:
            continue
        tr, te = train_test_split(np.arange(len(yS)), train_size=CFG['source_train_frac'],
                                  stratify=yS, random_state=seed)
        t0 = time.time(); mdl = make_model(mname, seed).fit(XS[tr], yS[tr]); fit_s = time.time() - t0
        classes = np.asarray(mdl.classes_).astype(str)
        rows, trows = [], []
        m = panel(yS[te], mdl.predict_proba(XS[te]), classes)
        rows.append(dict(model=mname, seed=seed, regime='in_domain', method='none', frac=0.0, n_eval=len(te),
                         n_buffer=0, buffer_benign=0, single_class_buffer=False, fit_s=round(fit_s, 1), **m))
        for r, order in ORD.items():
            P = mdl.predict_proba(XT[order]); yv = yT[order]; nat = nT[order]
            base = dict(model=mname, seed=seed, regime=r, method='none', frac=0.0)
            rows.append(dict(**base, n_eval=len(order), n_buffer=0, buffer_benign=0,
                             single_class_buffer=False, fit_s=round(fit_s, 1), **panel(yv, P, classes)))
            trows += type_rows(base, nat, yv, P, classes)
            for frac in CFG['fracs']:
                buf, rest = time_ordered_buffer(np.arange(len(order)), frac)
                nb = int(is_benign(yv[buf]).sum()); single = len(set(yv[buf])) < len(classes)
                b0 = dict(model=mname, seed=seed, regime=r, method='none', frac=frac)
                rows.append(dict(**b0, n_eval=len(rest), n_buffer=len(buf), buffer_benign=nb,
                                 single_class_buffer=single, fit_s=round(fit_s, 1),
                                 **panel(yv[rest], P[rest], classes)))
                if frac == CFG['fracs'][-1]:
                    trows += type_rows(b0, nat[rest], yv[rest], P[rest], classes)
                for meth in CFG['methods']:
                    b1 = dict(model=mname, seed=seed, regime=r, method=meth, frac=frac)
                    if single:
                        rows.append(dict(**b1, n_eval=len(rest), n_buffer=len(buf), buffer_benign=nb,
                                         single_class_buffer=True, fit_s=round(fit_s, 1),
                                         **{k: np.nan for k in METRICS}))
                        continue
                    Pc = calibrate(P[buf], yv[buf], P[rest], classes, meth)
                    rows.append(dict(**b1, n_eval=len(rest), n_buffer=len(buf), buffer_benign=nb,
                                     single_class_buffer=False, fit_s=round(fit_s, 1),
                                     **panel(yv[rest], Pc, classes)))
                    if frac == CFG['fracs'][-1]:
                        trows += type_rows(b1, nat[rest], yv[rest], Pc, classes)
            del P; gc.collect()
        append(B_CSV, rows, B_COLS); append(B_TYP, trows, T_COLS)
        df_ = pd.DataFrame(rows)
        for r in ORD:
            z = df_[(df_.regime == r) & (df_.method == 'none') & (df_.frac == 0)].iloc[0]
            c = df_[(df_.regime == r) & (df_.method == 'sigmoid') & (df_.frac == CFG['fracs'][-1])].iloc[0]
            print(f"{mname:5s} seed {seed} {r}: zero-shot AUROC {z.bin_auroc:.3f} MCC {z.mcc:.3f} | "
                  f"Platt 10% AUROC {c.bin_auroc:.3f} MCC {c.mcc:.3f} ECE {c.ece:.3f}")
        del mdl; gc.collect()
print('Stage 4 complete')

## Stage 5 — Drift per capture day

PSI and KS on every common-core feature between the source (shared families) and each target day,
for all traffic, benign only and attacks only. Each side is subsampled to at most 300,000 rows with a
fixed seed. Writes `conf_drift_features.csv` and `conf_drift_summary.csv`.

In [ ]:
rng = np.random.default_rng(0)
def sample(idx, n=CFG['drift_sample']):
    return idx if len(idx) <= n else rng.choice(idx, n, replace=False)
ben_S = is_benign(yS); feat_rows, summ = [], []
for r, rows in REG_ROWS.items():
    ben_T = is_benign(yT[rows])
    for subset, iS, iT in [('all', np.arange(len(yS)), rows),
                           ('benign', np.where(ben_S)[0], rows[ben_T]),
                           ('attack', np.where(~ben_S)[0], rows[~ben_T])]:
        a, b = XS[sample(iS)], XT[sample(iT)]
        ps = [psi(a[:, j], b[:, j]) for j in range(len(CORE))]
        ks = [ks_stat(a[:, j], b[:, j]) for j in range(len(CORE))]
        feat_rows += [dict(regime=r, subset=subset, feature=f, psi=p, ks=k) for f, p, k in zip(CORE, ps, ks)]
        ps = np.asarray(ps); ks = np.asarray(ks)
        summ.append(dict(regime=r, subset=subset, n_features=len(CORE), n_psi_gt_025=int((ps > 0.25).sum()),
                         median_psi=float(np.nanmedian(ps)), median_ks=float(np.nanmedian(ks)),
                         max_ks=float(np.nanmax(ks))))
pd.DataFrame(feat_rows).to_csv(f'{OUT}/conf_drift_features.csv', index=False)
drift_summary = pd.DataFrame(summ)
drift_summary.to_csv(f'{OUT}/conf_drift_summary.csv', index=False)
print(drift_summary.round(3).to_string(index=False))

## Stage 6 — Reproduction check against the preprint

Random Forest, seed 0. Within-dataset rows come from Stage 3. For transfer, the preprint trained on
the whole source, so this stage fits that model once and repeats zero-shot and Platt-10% scoring per day.
Writes `conf_reproduction.csv` with the difference in the `diff` column.

In [ ]:
rep = []
pre_A = pd.read_csv(f'{BASE}/results/demoA_CSE-CIC-IDS2018_rf.csv', index_col=0)
a = pd.read_csv(A_CSV)
a = a[(a.variant == 'native') & (a.model == 'rf') & (a.seed == 0)].set_index('split')
for pre_name, new_name in [('random', 'random'), ('temporal_stratified', 'time_all_seen'), ('chronological_blind', 'time_global')]:
    for k in ('macro_f1', 'mcc', 'fp_rate'):
        rep.append(dict(part='within', condition=new_name, metric=k, preprint=pre_A.loc[pre_name, k], now=a.loc[new_name, k]))

mdl = make_model('rf', 0).fit(XS, yS); classes = np.asarray(mdl.classes_).astype(str)
pre_B = {'regime_1': pd.read_csv(f'{BASE}/results/demoB_transfer_rf.csv', index_col=0),
         'regime_2': pd.read_csv(f'{BASE}/results/demoB_transfer_regime2_rf.csv', index_col=0)}
for r, order in ORD.items():
    P = mdl.predict_proba(XT[order]); yv = yT[order]
    z = panel(yv, P, classes)
    buf, rest = time_ordered_buffer(np.arange(len(order)), 0.10)
    c = panel(yv[rest], calibrate(P[buf], yv[buf], P[rest], classes, 'sigmoid'), classes)
    for k in ('mcc', 'auroc', 'ece'):
        rep.append(dict(part='transfer', condition=f'{r} zero-shot', metric=k, preprint=pre_B[r].loc['zero_shot', k], now=z[k]))
        rep.append(dict(part='transfer', condition=f'{r} Platt 10%', metric=k, preprint=pre_B[r].loc['buffer_10pct', k], now=c[k]))
rep = pd.DataFrame(rep); rep['diff'] = rep['now'] - rep['preprint']
rep.to_csv(f'{OUT}/conf_reproduction.csv', index=False)
print(rep.round(4).to_string(index=False))
del mdl; _ = gc.collect()

## Stage 7 — Summary tables

Means and standard deviations over seeds. Writes `conf_demoA_summary.csv`, `conf_unseen_summary.csv`,
`conf_demoB_summary.csv` and `conf_types_summary.csv`, and prints the headline numbers.

In [ ]:
def agg(df, keys, cols):
    g = df.groupby(keys)[cols].agg(['mean', 'std'])
    g.columns = [f'{a}_{b}' for a, b in g.columns]
    return g.reset_index()

A = pd.read_csv(A_CSV).drop_duplicates(A_KEYS, keep='last')
AC = pd.read_csv(A_CLS).drop_duplicates(A_KEYS + ['label'], keep='last')
B = pd.read_csv(B_CSV).drop_duplicates(B_KEYS, keep='last')
T = pd.read_csv(B_TYP).drop_duplicates(B_KEYS + ['attack_type'], keep='last')
cols = ['macro_f1', 'mcc', 'attack_recall', 'fp_rate', 'bin_auroc', 'bin_auprc', 'ece']
SA = agg(A, ['variant', 'split', 'model'], cols); SA.to_csv(f'{OUT}/conf_demoA_summary.csv', index=False)
U = AC[(AC.variant == 'native') & (AC.split == 'time_global') & (~AC.in_train)]
SU = agg(U, ['label', 'model'], ['flagged_attack', 'recall', 'n_test'])
tops = U.groupby(['label', 'model'])['top_pred'].agg(lambda s: s.value_counts().index[0]).rename('top_pred').reset_index()
SU = SU.merge(tops, on=['label', 'model']); SU.to_csv(f'{OUT}/conf_unseen_summary.csv', index=False)
SB = agg(B, ['regime', 'model', 'method', 'frac'], cols + ['auroc', 'auprc', 'brier']); SB.to_csv(f'{OUT}/conf_demoB_summary.csv', index=False)
ST = agg(T, ['regime', 'method', 'frac', 'attack_type', 'model'], ['auroc_vs_benign', 'flagged_attack', 'n'])
ST.to_csv(f'{OUT}/conf_types_summary.csv', index=False)

print('Within-dataset (native labels), mean over seeds:')
print(SA[SA.variant == 'native'][['split', 'model', 'macro_f1_mean', 'attack_recall_mean', 'fp_rate_mean', 'bin_auroc_mean']].round(3).to_string(index=False))
print('\nUnseen attack types under the global time split (alarm rate = flagged as any attack):')
print(SU[['label', 'model', 'flagged_attack_mean', 'top_pred', 'n_test_mean']].round(3).to_string(index=False))
print('\nTransfer, zero-shot (all rows of the day) and 10% buffer, mean over seeds:')
show = SB[((SB.method == 'none') & (SB.frac == 0)) | (SB.frac == CFG['fracs'][-1])]
print(show[['regime', 'model', 'method', 'frac', 'bin_auroc_mean', 'bin_auprc_mean', 'mcc_mean', 'ece_mean', 'fp_rate_mean', 'attack_recall_mean']].round(3).to_string(index=False))
print('\nSingle-class buffers:', int(B.single_class_buffer.sum()))

## Stage 8 — Figures (600 dpi, PNG and PDF)

Sized for the IEEE two-column layout, with the paper's own names on every axis and legend.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'serif', 'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7,
                     'legend.fontsize': 6.5, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.bbox': 'tight'})
COL = {'rf': '#1b6ca8', 'lgbm': '#e07b39', 'mlp': '#3a923a'}
def save(fig, name):
    fig.savefig(f'{FIG}/{name}.png', dpi=CFG['dpi']); fig.savefig(f'{FIG}/{name}.pdf'); plt.close(fig)
    print('saved', name)

# Fig: split protocol (class score vs attack alarm)
nat = SA[SA.variant == 'native']
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.1))
w = 0.26; xs = np.arange(len(CFG['splits']))
for ax, (metric, title) in zip(axes, [('macro_f1', '(a) Class-by-class score (macro-F1)'),
                                       ('attack_recall', '(b) Attacks raised as an alarm (any attack class)')]):
    for i, mname in enumerate(CFG['models']):
        sub = nat[nat.model == mname].set_index('split').loc[CFG['splits']]
        ax.bar(xs + (i - 1) * w, sub[f'{metric}_mean'], w, yerr=sub[f'{metric}_std'].fillna(0),
               color=COL[mname], label=MODEL_NAME[mname], capsize=1.5, error_kw={'lw': 0.6})
    ax.set_xticks(xs); ax.set_xticklabels(['Random split', 'Time split,\nall classes\nseen', 'Time split,\nlater attacks\nunseen'])
    ax.set_ylim(0, 1.05); ax.set_title(title, loc='left')
axes[0].set_ylabel('Score')
h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.1))
save(fig, 'fig_conf_split_protocol')

# Fig: unseen attack types
ul = SU.groupby('label')['flagged_attack_mean'].mean().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(3.5, 2.2)); ys = np.arange(len(ul)); h = 0.26
for i, mname in enumerate(CFG['models']):
    sub = SU[SU.model == mname].set_index('label').reindex(ul)
    ax.barh(ys + (i - 1) * h, sub['flagged_attack_mean'], h, xerr=sub['flagged_attack_std'].fillna(0),
            color=COL[mname], label=MODEL_NAME[mname], capsize=1.2, error_kw={'lw': 0.5})
ax.set_yticks(ys); ax.set_yticklabels(ul); ax.set_xlim(0, 1.0)
ax.set_xlabel('Share of flows raised as an attack'); ax.legend(frameon=False, loc='lower right')
save(fig, 'fig_conf_unseen_attacks')

# Fig: transfer, ranking (AUROC) vs decision (MCC) against buffer size
fig, axes = plt.subplots(2, 2, figsize=(7.0, 3.6), sharex=True)
xb = [0] + [int(f * 100) for f in CFG['fracs']]
for j, r in enumerate(['regime_1', 'regime_2']):
    for i, (metric, lab) in enumerate([('bin_auroc', 'Ranking: AUROC'), ('mcc', 'Decisions: MCC')]):
        ax = axes[i, j]
        for mname in CFG['models']:
            for meth, ls in [('sigmoid', '-'), ('isotonic', '--')]:
                vals, errs = [], []
                z = SB[(SB.regime == r) & (SB.model == mname) & (SB.method == 'none') & (SB.frac == 0)]
                vals.append(z[f'{metric}_mean'].iloc[0]); errs.append(z[f'{metric}_std'].fillna(0).iloc[0])
                for f in CFG['fracs']:
                    s = SB[(SB.regime == r) & (SB.model == mname) & (SB.method == meth) & (SB.frac == f)]
                    vals.append(s[f'{metric}_mean'].iloc[0]); errs.append(s[f'{metric}_std'].fillna(0).iloc[0])
                ax.errorbar(xb, vals, yerr=errs, color=COL[mname], ls=ls, marker='o', ms=2.5, lw=0.9, capsize=1.5,
                            label=f'{MODEL_NAME[mname]}, {METHOD_NAME[meth]}')
        ax.set_title(f"{'Regime 1' if r == 'regime_1' else 'Regime 2'}: {lab}", loc='left')
        ax.set_xticks(xb)
        if i == 1: ax.set_xlabel('Labelled target flows used for recalibration (%)')
        if j == 0: ax.set_ylabel(lab.split(': ')[1])
h_, l_ = axes[0, 0].get_legend_handles_labels()
fig.legend(h_, l_, ncol=3, frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.08))
save(fig, 'fig_conf_transfer')

# Fig: per attack type ranking, zero-shot
z = ST[(ST.method == 'none') & (ST.frac == 0)]
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.2), sharex=True)
for ax, r in zip(axes, ['regime_1', 'regime_2']):
    s = z[z.regime == r].groupby('attack_type')['auroc_vs_benign_mean'].agg(['mean', 'std']).sort_values('mean')
    ax.barh(np.arange(len(s)), s['mean'], xerr=s['std'].fillna(0), color='#5a6f8c', capsize=1.2, error_kw={'lw': 0.5})
    ax.axvline(0.5, color='grey', lw=0.6, ls=':')
    ax.set_yticks(np.arange(len(s))); ax.set_yticklabels(s.index)
    ax.set_title(f"{'Regime 1' if r == 'regime_1' else 'Regime 2'}: zero-shot AUROC per attack type", loc='left')
    ax.set_xlim(0, 1); ax.set_xlabel('AUROC against benign flows (mean over models and seeds)')
save(fig, 'fig_conf_attack_types')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "21: conference extension (3 models x 3 seeds, unseen-attack alarms, ranking vs decisions under Platt and isotonic, per-type ranking, per-day drift, label audit, reproduction check)"],
  capture_output=True, text=True)
print(r.stdout)
print(r.stderr)